In [1]:
import pickle as pkl
import pandas as pd

In [2]:
df = pd.read_parquet("yellow_tripdata_2025-01_clean.parquet")
num_cols = df.select_dtypes(include='number').columns   # collecting column which has datatype number and assing those column names in a variable
df1 = df[(df[num_cols] >= 0).all(axis=1)]   # removing negative value records
df1=df1.sample(100000)  # Get 100,000 record samples instead of all population which has over 3 Million

In [3]:
# splitting the dataset independent variable and dependent variable
X_inde=df1[['passenger_count', 'trip_distance', 'RatecodeID', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax',
       'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'Airport_fee', 'cbd_congestion_fee',
        'store_and_fwd_flag_Y']]

y_dep=df1[['trip_duration']]

In [7]:
# Function for best future from independent variable using Select K function
def selectkbest(indep_X, dep_Y, n):
    from sklearn.feature_selection import SelectKBest, f_regression
    
    test = SelectKBest(score_func=f_regression, k=n)
    fit1 = test.fit(indep_X, dep_Y)
    
    selectk_features = fit1.transform(indep_X)
    selected_columns = indep_X.columns[fit1.get_support()]
    return selectk_features, selected_columns

In [8]:
kbest,selected_columns =selectkbest(X_inde,y_dep,6)

C:\Anaconda3\envs\aiml\Lib\site-packages\sklearn\utils\validation.py:1406: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [9]:
# Function for standardizing the data and splitting Training set and test set.
def split_scalar(indep_X,dep_Y):
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler
    X_train, X_test, y_train, y_test = train_test_split(indep_X, dep_Y, test_size = 0.25, random_state = 0)
    sc = StandardScaler()
    X_train = sc.fit_transform(X_train)
    X_test = sc.transform(X_test)    
    return X_train, X_test, y_train, y_test

In [10]:
X_train,X_test,y_train,y_test=split_scalar(kbest,y_dep)  # passing selectk output as input here

In [11]:
loaded_model = pkl.load(open("Fleet_Efficiency.sav",'rb'))

In [12]:
loaded_columns = pkl.load(open("columns.sav",'rb'))

In [13]:
X_inde=df1[['passenger_count', 'trip_distance', 'RatecodeID', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax',
       'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'Airport_fee', 'cbd_congestion_fee',
        'store_and_fwd_flag_Y']]

In [14]:
input = {
    'passenger_count': 2,
    'trip_distance': 10,
    'RatecodeID': 1,
    'PULocationID': 177,
    'DOLocationID': 198,
    'payment_type': 1,
    'fare_amount': 20,
    'extra': 1,
    'mta_tax': 0,
    'tip_amount': 2,
    'tolls_amount': 2,
    'improvement_surcharge': 1.2,
    'total_amount': 27,
    'congestion_surcharge': 1,
    'Airport_fee': 0,
    'cbd_congestion_fee': 0,
    'store_and_fwd_flag': 'N'
}

In [15]:
input_df=pd.DataFrame([input])

In [16]:
input_df.columns

Index(['passenger_count', 'trip_distance', 'RatecodeID', 'PULocationID',
       'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax',
       'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount',
       'congestion_surcharge', 'Airport_fee', 'cbd_congestion_fee',
       'store_and_fwd_flag'],
      dtype='object')

In [17]:
input_df

,passenger_count,trip_distance,RatecodeID,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,store_and_fwd_flag
0,2,10,1,177,198,1,20,1,0,2,2,1.2,27,1,0,0,N


In [18]:
input_df=pd.get_dummies(input_df,dtype=int)

In [19]:
input_df = input_df.reindex(columns=loaded_columns, fill_value=0)

In [20]:
input_df

,passenger_count,trip_distance,RatecodeID,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,store_and_fwd_flag_Y
0,2,10,1,177,198,1,20,1,0,2,2,1.2,27,1,0,0,0


In [21]:
input_selected = input_df[selected_columns]

In [22]:
loaded_model.predict(input_selected)

C:\Anaconda3\envs\aiml\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(


array([29.93916667])